In [2]:
import requests
import json
import pandas as pd
import numpy as np
import os

In [8]:
data = json.loads(open("keys.json").read())

API_KEY = data["STEAM_WEB_API"] 

BASE_URL = "https://www.steamwebapi.com/steam/api/"


def call(method: str, call_path: str, extra_params: str | None = None, payload: dict | None = None):
    """
    calls the API, stores the response to /data, returns (location of saved file, errorcode)
    """
    method = method.upper()
    body = ""
    headers = {}

    request_url = BASE_URL + call_path + "?key=" + API_KEY 
    output_path = f"data/{call_path.replace('/', '_')}"

    if extra_params != None:
        request_url = request_url + "&" + extra_params
        output_path = output_path + "_" + extra_params + ".json"

    if payload:
        body = json.dumps(payload)


    if method != "GET" and payload:
        headers["Content-Type"] = "application/json"

    try:
        r = requests.request(
            method,
            request_url,
            headers=headers,
            data=body.encode() if body else None,
            timeout=15,
        )
        r.raise_for_status()
        data_json = r.json()

        if len(data_json) == 0:
            return output_path, "RESPONSE EMPTY"

        file_opening_mode = "x"

        if (os.path.exists(output_path)):
            file_opening_mode = "w"
        
        with open(output_path, file_opening_mode) as f:
            json.dump(data_json, f)

        return output_path, None
    except requests.RequestException as e:
        detail = e.response.text if e.response is not None else "no body"
        return output_path, f"{e} — {detail}"
    

In [ ]:
call_path = "items"
game_param = "game=cs2" 

payload = {
    "markets":",,,"
}

for pagenum in range(0,5):
    full_params = game_param + '&' + 'page=' + str(pagenum)
    location, errorcode = call("GET", call_path, full_params)
    if (errorcode != None):
        print(errorcode)
        break
    print(f"Nr: {pagenum}, File: {location}" +  ", Size: " + str(os.path.getsize(location)/1000) + "MB" )

Nr: 0, File: data/items_game=cs2&page=0.json, Size: 268068.624MB
Nr: 1, File: data/items_game=cs2&page=1.json, Size: 268068.624MB


FileNotFoundError: [Errno 2] No such file or directory: 'data/items_game=cs2&page=2.json'